In [ ]:
%cd /content

!rm -rf /content/grobid /root/.gradle /root/.cache/gradle

!apt-get install -y openjdk-17-jdk-headless > /dev/null 2>&1

import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PATH'] = f"{os.environ['JAVA_HOME']}/bin:" + os.environ['PATH']

!java -version

!git clone --depth=1 --branch 0.8.2 https://github.com/kermitt2/grobid.git /content/grobid
%cd /content/grobid
!chmod +x ./gradlew

print("\nGROBID скачан, готов к сборке и запуску")

/content
openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)
Cloning into '/content/grobid'...
remote: Enumerating objects: 9311, done.
remote: Counting objects: 100% (9311/9311), done.
remote: Compressing objects: 100% (4406/4406), done.
remote: Total 9311 (delta 5230), reused 7961 (delta 4683), pack-reused 0 (from 0)
Receiving objects: 100% (9311/9311), 296.84 MiB | 18.84 MiB/s, done.
Resolving deltas: 100% (5230/5230), done.
Note: switching to '9088cf1705d8b695bb0a8b2ab1519a96d669bcdc'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new

In [ ]:
import subprocess
import time
import requests

log_file = open('/content/grobid.log', 'w')
# Передаём JAVA_HOME явно, чтобы дочерний процесс точно использовал Java 17
env = os.environ.copy()
env['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
env['PATH'] = f"{env['JAVA_HOME']}/bin:" + env['PATH']

proc = subprocess.Popen(
    ['./gradlew', 'run'],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    cwd='/content/grobid',
    env=env,
)
print(f"GROBID запущен в фоне, PID={proc.pid}")
print("Ожидаю готовности\n")

max_wait_seconds = 900
elapsed = 0
ready = False
while elapsed < max_wait_seconds:
    try:
        r = requests.get('http://localhost:8070/api/isalive', timeout=3)
        if r.status_code == 200:
            ready = True
            break
    except requests.exceptions.RequestException:
        pass

    if proc.poll() is not None:
        print("GROBID процесс упал! Смотри /content/grobid.log")
        break

    time.sleep(15)
    elapsed += 15
    print(f"  ждём... прошло {elapsed} сек")

if ready:
    print(f"\nGROBID готов на http://localhost:8070 (за {elapsed} сек)")
else:
    print("\nGROBID не поднялся за отведённое время. Проверить /content/grobid.log")

GROBID запущен в фоне, PID=50843
Ожидаю готовности

  ждём... прошло 15 сек
  ждём... прошло 30 сек
  ждём... прошло 45 сек
  ждём... прошло 60 сек
  ждём... прошло 75 сек
  ждём... прошло 90 сек
  ждём... прошло 105 сек
  ждём... прошло 120 сек
  ждём... прошло 135 сек
  ждём... прошло 150 сек

GROBID готов на http://localhost:8070 (за 150 сек)


In [ ]:
import subprocess
import time
import re

!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /contenфt/cloudflared

tunnel_log = '/content/cloudflared.log'
tunnel_proc = subprocess.Popen(
    ['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8070'],
    stdout=open(tunnel_log, 'w'),
    stderr=subprocess.STDOUT,
)
print(f"cloudflared запущен, PID={tunnel_proc.pid}")
print("Ожидаем появления публичного URL...\n")

public_url = None
for attempt in range(20):
    time.sleep(2)
    try:
        with open(tunnel_log) as f:
            log_content = f.read()
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log_content)
        if match:
            public_url = match.group(0)
            break
    except FileNotFoundError:
        pass

if public_url:
    print(f"\n{'='*60}")
    print(f"GROBID URL для основного ноутбука:")
    print(f"  {public_url}")
    print(f"{'='*60}")
    print(f"страница 'Grobid Web Application'.")
else:
    print("Не удалось получить URL. Смотреть /content/cloudflared.log")
    !tail -30 /content/cloudflared.log

chmod: cannot access '/contenфt/cloudflared': No such file or directory
cloudflared запущен, PID=51633
Ожидаем появления публичного URL...


GROBID URL для основного ноутбука:
  https://near-about-parker-steady.trycloudflare.com
страница 'Grobid Web Application'.
